# Binding Energy of a Pt₄ Cluster on Monolayer MoS₂

> **W. A. Saidi**
> "Density Functional Theory Study of Nucleation and Growth of Pt Nanoparticles on MoS2(001) Surface"
> Cryst. Growth Des. 15, 642 (2015)
> [DOI: 10.1021/cg5013395](https://doi.org/10.1021/cg5013395)

Computes the binding energy per Pt atom of the t-(Mo)₄-S Pt₄ cluster and of one Pt atom on top of Mo, on the 5x5 MoS₂
cells created in the [structure notebook](defect_point_adatom_island_molybdenum_disulfide_platinum.ipynb), with Saidi's
Eq. 1, E_b = (E(MoS₂ + Pt_n) - E(MoS₂) - n E(Pt atom)) / n, and compares both with Saidi's Table 1.

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](defect_point_adatom_island_molybdenum_disulfide_platinum.ipynb), which saves `MoS2 5x5 monolayer`, `MoS2 5x5 Pt1 atop Mo` and `MoS2 5x5 Pt4 t-(Mo)4-S` to the `uploads` folder.
1. Set the material names and parameters in cells 1.2-1.4 (or use the defaults); each relaxed slab is saved as `<name> relaxed <relaxation model tag>`.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, model and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the three slabs by name, build the isolated Pt atom in a cubic box, print provenance, then save all four to the platform.
1. Configure the shared DFT model: one model for every workflow below.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Relax the three slabs at fixed cell with the dipole correction, reusing a relaxation job that already exists.
1. Run one spin-polarized total energy job on the isolated Pt atom, and read the slab energies from the relaxation jobs.
1. Retrieve the results: the binding energies of Pt₄ and Pt₁ and the mean Pt bond lengths in the relaxed cluster.
1. Compare with Saidi (2015).

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")


### 1.2. Materials

In [ ]:
# Names saved by defect_point_adatom_island_molybdenum_disulfide_platinum.ipynb.
PRISTINE_NAME = "MoS2 5x5 monolayer"
PT1_NAME = "MoS2 5x5 Pt1 atop Mo"
PT4_NAME = "MoS2 5x5 Pt4 t-(Mo)4-S"
PT_ATOM_NAME = "Pt atom"
PT_ATOM_BOX = 12.0  # Å, edge of the cubic box around the isolated Pt atom; paper: box size not stated


### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

TOTAL_ENERGY_SEARCH_TERM = "total_energy.json"
RELAX_WORKFLOW_SEARCH_TERM = "fixed_cell_relaxation.json"
APPLICATION_NAME = "espresso"

CLUSTER_NAME = None
QUEUE_NAME = QueueName.OR
PPN = 16  # OR allows 16 cores per job
TIME_LIMIT = "04:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

# Saidi 2015, Table 1: E_b in eV per Pt atom; mean t-(Mo)4-S bond lengths in Å, over pairs closer than BOND_CUTOFF
SAIDI = {"e_b_pt4": -3.43, "e_b_pt1": -2.83, "d_pt_mo": 2.91, "d_pt_s": 2.35, "d_pt_pt": 2.61}
BOND_CUTOFF = 3.0  # Å


### 1.4. DFT model parameters

In [ ]:
# paper: VASP PAW at 350 eV; ours GBRV ultrasoft at its own converged 40/200 Ry
FUNCTIONAL = "pbe"
PSEUDOPOTENTIAL_TYPE = "us"
ECUTWFC = 40   # Ry
ECUTRHO = 200  # Ry

GAMMA_KGRID = [1, 1, 1]  # paper: Γ only, for the slabs and the isolated atom
# paper: Fermi-Dirac, k_B T = 0.05 eV = 0.003675 Ry
SMEARING_SETTINGS = {"occupations": "smearing", "smearing": "fermi-dirac", "degauss": 0.003675}
# paper: dipole correction along z; eamp = 0 keeps the sawtooth a correction, no external field
DIPOLE_SETTINGS = {"control": {"tefield": True, "dipfield": True}, "system": {"edir": 3, "eamp": 0.0, "eopreg": 0.05}}
# paper: the isolated Pt atom spin-polarized with two unpaired electrons; the slabs spin-averaged
SPIN_SETTINGS = {"nspin": 2, "tot_magnetization": 2}
# paper: all atoms relaxed to 0.005 eV/Å. nstep ends each relaxation inside TIME_LIMIT;
# the residual force is printed in section 7. To reach the threshold, raise nstep (and TIME_LIMIT, cluster permitting).
RELAXATION_SETTINGS = {"forc_conv_thr": 1.9e-4, "nstep": 10}
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry G fd{SMEARING_SETTINGS['degauss']}"

SCF_UNIT = "pw_scf"
RELAX_UNIT = "pw_relax"
# Names the relaxation jobs; the slab energies in Eq. 1 are read from these jobs.
RELAX_TAG = f"{MODEL_TAG} dip f{RELAXATION_SETTINGS['forc_conv_thr']} n{RELAXATION_SETTINGS['nstep']}"


## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()


### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account

In [ ]:
client.list_accounts()


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")


## 3. Load the materials
### 3.1. Load the slabs from the uploads folder or the platform, build the isolated Pt atom, and print provenance

In [ ]:
from collections import Counter
from ase import Atoms
from mat3ra.made.material import Material
from mat3ra.made.tools.convert import from_ase
from mat3ra.notebooks_utils.core.entity.material.api import load_material

slabs = {name: load_material(client, FOLDER, name, ACCOUNT_ID) for name in (PRISTINE_NAME, PT1_NAME, PT4_NAME)}
for slab in slabs.values():
    slab.basis.set_labels_from_list([])
pt_atom = Material.create(from_ase(Atoms("Pt", cell=[PT_ATOM_BOX] * 3, pbc=True)))
pt_atom.name = PT_ATOM_NAME
materials_by_name = {**slabs, PT_ATOM_NAME: pt_atom}

for name, material in materials_by_name.items():
    a, b, c = material.lattice.a, material.lattice.b, material.lattice.c
    composition = "".join(f"{e}{n}" for e, n in sorted(Counter(material.basis.elements.values).items()))
    print(f"{name}: {composition}, {material.basis.number_of_atoms} atoms, "
          f"cell {a:.2f} x {b:.2f} x {c:.2f} Å")


### 3.2. Save the materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = {name: get_or_create_material(client, material, ACCOUNT_ID)
                   for name, material in materials_by_name.items()}


## 4. Configure the shared DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype="gga", functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")


## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")


### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")


## 6. Fixed-cell relaxation of the three slabs

Each slab is relaxed with the dipole correction in the middle of its vacuum; the final total energy of
each relaxation is the slab energy in Eq. 1. A relaxation job with the same settings on the same slab is reused.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, apply_scf_kgrid, patch_workflow_qe_input
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material

relaxation_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    RELAX_WORKFLOW_SEARCH_TERM
)
relax_job_ids = {}
new_job_ids = []
for name, slab in slabs.items():
    workflow = Workflow.create(relaxation_workflow_config)
    workflow.name = f"Fixed-cell Relaxation {name} {RELAX_TAG}"
    workflow.subworkflows[0].model = model
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=RELAX_UNIT)
    apply_scf_kgrid(workflow, GAMMA_KGRID, material=slab, unit_name=RELAX_UNIT)
    patch_workflow_qe_input(workflow, {"control": RELAXATION_SETTINGS}, [RELAX_UNIT])
    patch_workflow_qe_input(workflow, DIPOLE_SETTINGS, [RELAX_UNIT])
    # emaxpos: the middle of the vacuum above the slab, in crystal z
    emaxpos = (max(coordinate[2] for coordinate in slab.basis.coordinates.values) + 1) / 2
    patch_workflow_qe_input(workflow, {"system": {**SMEARING_SETTINGS, "emaxpos": emaxpos}}, [RELAX_UNIT])
    job = find_job_for_material(client, saved_materials[name]["_id"], workflow.name, ACCOUNT_ID,
                                statuses=("submitted", "queued", "active", "finished"),
                                kgrid=GAMMA_KGRID, unit_name=RELAX_UNIT)
    if job is None:
        job = create_job(
            api_client=client, materials=[saved_materials[name]], workflow=workflow, project_id=project_id,
            owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
        )
        new_job_ids.append(job["_id"])
        print(f"✅ {name}: created relaxation job {job['_id']}")
    else:
        print(f"♻️  {name}: reusing existing relaxation job {job['_id']}")
    relax_job_ids[name] = job["_id"]


In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

if new_job_ids:
    submit_jobs(client.jobs, new_job_ids)
    print(f"✅ Submitted {len(new_job_ids)} relaxation job(s).")
if relax_job_ids:
    await wait_for_jobs_to_finish_async(client.jobs, list(relax_job_ids.values()), poll_interval=POLL_INTERVAL)


## 7. Total energies
### 7.1. Isolated Pt atom

In [ ]:
total_energy_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    TOTAL_ENERGY_SEARCH_TERM
)
workflow = Workflow.create(total_energy_workflow_config)
workflow.name = (f"Total Energy {PT_ATOM_NAME} {MODEL_TAG} "
                 f"nspin{SPIN_SETTINGS['nspin']} m{SPIN_SETTINGS['tot_magnetization']}")
workflow.subworkflows[0].model = model
apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=SCF_UNIT)
apply_scf_kgrid(workflow, GAMMA_KGRID, material=pt_atom)
patch_workflow_qe_input(workflow, {"system": {**SMEARING_SETTINGS, **SPIN_SETTINGS}}, [SCF_UNIT])
pt_atom_job = find_job_for_material(client, saved_materials[PT_ATOM_NAME]["_id"], workflow.name, ACCOUNT_ID,
                                    statuses=("submitted", "queued", "active", "finished"), kgrid=GAMMA_KGRID)
if pt_atom_job is None:
    pt_atom_job = create_job(
        api_client=client, materials=[saved_materials[PT_ATOM_NAME]], workflow=workflow, project_id=project_id,
        owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
    )
    submit_jobs(client.jobs, [pt_atom_job["_id"]])
    print(f"✅ {PT_ATOM_NAME}: created and submitted Total Energy job {pt_atom_job['_id']}")
else:
    print(f"♻️  {PT_ATOM_NAME}: reusing existing Total Energy job {pt_atom_job['_id']}")


In [ ]:
await wait_for_jobs_to_finish_async(client.jobs, [pt_atom_job["_id"]], poll_interval=POLL_INTERVAL)


### 7.2. Relaxed slabs

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.core.entity.material.api import get_final_structure_for_job

e_pt_atom = get_properties_for_job(client, pt_atom_job["_id"], "total_energy")[0]["value"]
relaxed_materials = {}
slab_energies = {}
for name, job_id in relax_job_ids.items():
    relaxed_material = get_final_structure_for_job(client, job_id)
    client.materials.update(relaxed_material.id, {"name": f"{name} relaxed {RELAX_TAG}"})
    relaxed_materials[name] = relaxed_material
    slab_energies[name] = get_properties_for_job(client, job_id, "total_energy")[0]["value"]
    total_force = get_properties_for_job(client, job_id, "total_force")[0]
    print(f"{name}: residual force {total_force['value']:.4f} {total_force['units']}")


## 8. Retrieve the results
### 8.1. Binding energies (Eq. 1)

In [ ]:
e_pristine, e_pt1, e_pt4 = (slab_energies[name] for name in (PRISTINE_NAME, PT1_NAME, PT4_NAME))
e_binding_pt4 = (e_pt4 - e_pristine - 4 * e_pt_atom) / 4
e_binding_pt1 = e_pt1 - e_pristine - e_pt_atom


### 8.2. Mean Pt bond lengths in the relaxed Pt₄ cluster

In [ ]:
import math

cluster_cartesian = relaxed_materials[PT4_NAME].clone()
cluster_cartesian.to_cartesian()
cluster_sites = list(zip(cluster_cartesian.basis.coordinates.values, cluster_cartesian.basis.elements.values))
platinum_sites = [site for site, element in cluster_sites if element == "Pt"]
for neighbor, paper_key in (("Mo", "d_pt_mo"), ("S", "d_pt_s"), ("Pt", "d_pt_pt")):
    distances = [math.dist(platinum, site) for platinum in platinum_sites
                 for site, element in cluster_sites if element == neighbor]
    bonds = [distance for distance in distances if 0 < distance < BOND_CUTOFF]
    print(f"Pt-{neighbor}: {sum(bonds) / len(bonds):.3f} Å, Saidi {SAIDI[paper_key]:.2f} Å")


## 9. Comparison with Saidi (2015)

In [ ]:
deviation_binding_pt4 = 100 * (e_binding_pt4 - SAIDI["e_b_pt4"]) / abs(SAIDI["e_b_pt4"])
deviation_binding_pt1 = 100 * (e_binding_pt1 - SAIDI["e_b_pt1"]) / abs(SAIDI["e_b_pt1"])
print(f"E_b(Pt4, t-(Mo)4-S): {e_binding_pt4:.3f} eV/Pt, "
      f"Saidi {SAIDI['e_b_pt4']:.2f} eV/Pt ({deviation_binding_pt4:+.1f} %)")
print(f"E_b(Pt1, atop Mo): {e_binding_pt1:.3f} eV/Pt, "
      f"Saidi {SAIDI['e_b_pt1']:.2f} eV/Pt ({deviation_binding_pt1:+.1f} %)")

## References

[1] Saidi, W. A. (2015). Density Functional Theory Study of Nucleation and Growth of Pt Nanoparticles on MoS2(001) Surface. Cryst. Growth Des., 15(2), 642–652. https://doi.org/10.1021/cg5013395